In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import torch
from torch import tensor as t
import matplotlib.pyplot as plt

In [ ]:
epochs = 100
num_classes = 1024
variance = .05

x = torch.linspace(1, epochs, epochs)
loss_values = 1 + (-.5*torch.log(t(x/num_classes)) / (torch.log(t(x+1)))) + variance * torch.randn(epochs)
print(loss_values.min())
# Plot the synthetic loss curve
plt.plot(loss_values.numpy(), label='Synthetic Loss Curve', color='blue')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
x  = loss_values
y = torch.tensor(x, dtype=torch.float32).view(-1, 1)
x = torch.tensor(torch.arange(x.shape[0]), dtype=torch.float32).view(-1, 1)

# Smooth the data using a simple moving average
# Smoothing constant
smoothing_constant = 0.92

# Smooth the data using a simple moving average with a smoothing constant
window_size = 3
conv_weights = (smoothing_constant / window_size) * torch.ones(window_size)
smoothed_y = torch.nn.functional.conv1d(y.view(1, 1, -1), conv_weights.view(1, 1, -1), padding=(window_size-1)//2).view(-1)

# Perform linear regression on the last 20 steps of smoothed_y
n = 20

lookback = 50

x_last_n = x[-n-lookback:-n]
print(x_last_n.shape)
y_last_n = smoothed_y[-n-lookback:-n]
X = torch.cat([torch.ones(lookback, 1), x_last_n], dim=1)
coefficients = torch.linalg.lstsq(X, y_last_n.view(-1, 1)).solution.flatten()
print(coefficients)

intercept, slope = coefficients[0], coefficients[1]

# Plot the original and smoothed data along with the regression line
plt.scatter(x.numpy(), y.numpy(), label='Original Data')
plt.plot(x.numpy(), smoothed_y.numpy(), label='Smoothed Data', color='orange')
plt.plot(x.numpy(), slope * x.numpy() + intercept.numpy(), label=f'Linear Regression: y = {slope:.2f}x + {intercept:.2f}', color='red')
plt.legend()
plt.show()

In [ ]:
from nanodrz.optim import calculate_smoothed_slope
calculate_smoothed_slope(loss_values.tolist()[-20:])